# Bluestock MF Capstone - Exploratory Data Analysis

This notebook explores the mutual fund datasets after ETL.

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect('../bluestock_mf.db')

fund = pd.read_sql('SELECT * FROM dim_fund', conn)
nav = pd.read_sql('SELECT * FROM fact_nav', conn, parse_dates=['date'])
perf = pd.read_sql('''
    SELECT f.scheme_name, f.fund_house, f.category, p.*
    FROM fact_performance p JOIN dim_fund f ON p.amfi_code = f.amfi_code
''', conn)
tx = pd.read_sql('SELECT * FROM fact_transactions', conn, parse_dates=['date'])
print('Tables loaded successfully')

## 1. Fund Master Overview

In [ ]:
fund.head()

In [ ]:
fund['category'].value_counts().plot(kind='bar', title='Schemes by Category')
plt.show()

## 2. NAV History

In [ ]:
sample = nav[nav['amfi_code'] == '125497']
sample.plot(x='date', y='nav', title='HDFC Top 100 NAV Trend')
plt.show()

## 3. Risk-Return Analysis

In [ ]:
sns.scatterplot(data=perf, x='std_dev_pct', y='return_1yr_pct', hue='category', s=100)
plt.title('Risk vs Return')
plt.show()

## 4. Investor Transactions

In [ ]:
tx.groupby('transaction_type')['amount'].sum().plot(kind='pie', autopct='%1.1f%%')
plt.title('Transaction Amount Distribution')
plt.ylabel('')
plt.show()

## 5. Top Performing Funds

In [ ]:
perf.nlargest(10, 'sharpe_ratio')[['scheme_name', 'category', 'sharpe_ratio', 'return_1yr_pct']]